# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is Our team's mission proposal. 

> **Team size:** 3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Leila Rahimiyadkuri||
| Member | Forough Asgari| |
| Member | Sara Davoodabadi | |


## 2. Mission Title & Research Question

**Title:**  
Predicting and Explaining Customer Churn in Retail Banking: Combining Causal Inference, Supervised Classification, and Customer Segmentation

**Research question:**  
Does a customer's account balance or credit score *causally* increase their probability of leaving the bank and if so, does this effect differ across distinct customer segments identified through unsupervised learning?

**Why it matters:**  
Banks typically spend far more to acquire a new customer than to keep an existing one. Most churn studies focus on prediction alone, they tell you *who* is likely to leave but not *why*. We want to go one step further: by combining a causal lens with predictive modeling and customer clustering, we hope to give actionable insight into which financial levers (like improving someone's credit score or offering a better savings rate) could realistically reduce churn. That distinction between correlation and causation matters a lot when a bank is deciding where to spend money on retention.

## 3. Data

**Source:**  
The Bank Customer Churn Dataset, an open-source retail banking dataset hosted on Kaggle.

Link: https://www.kaggle.com/datasets/shantanudhakadd/bank-customer-churn-prediction

**Unit of observation:** One row = one individual retail bank customer. 

The dataset contains 10,000 rows and 14 columns.

**Key variables:**

| Variable | Type | Role | Description |
|----------|------|------|-------------|
| RowNumber | Integer | - | Row index  - carries no information for using in models |
| CustomerId | Integer | - | identifier - carries no information for using in models |
| Surname | String | - | Customer's last name - carries no information for using in models |
| CreditScore | Integer | Treatment(W₁) | Credit score of the customer |
| Geography | Categorical | Feature | Country of residence: France, Germany, or Spain |
| Gender | Categorical | Feature | Customer's gender (Male / Female) |
| Age | Integer | Confounder(X₁) | The age of the customer |
| Tenure | Integer | Feature | Number of years the customer has been with the bank (0–10) |
| Balance | Float | Treatment (W₂) | Current account balance|
| NumOfProducts | Integer | Feature | Number of bank product facilities customer is using |
| HasCrCard | Binary (0/1) | Feature | Whether the customer holds a credit card (1 = yes) |
| IsActiveMember | Binary (0/1) | Feature | Whether the customer has been actively transacting (1 = active) |
| EstimatedSalary | Float | Confounder(X₂) | Annual estimated salary |
| Exited | Binary (0/1) | Target (Y) | Whether the customer closed their account (1 = churned, 0 = stayed) |

**Potential data quality issues:**  
The most notable issue is class imbalance: in most retail churn datasets, churners (Exited = 1) make up only around 20% of records. We will not oversample during training, but we will use class-weighted models and report F1-Score and AUC-ROC rather than raw accuracy to avoid being misled by a majority-class baseline. We will also check for any zero-balance customers, who may represent inactive accounts rather than genuine churn risk, and handle them explicitly in the analysis.

In [ ]:
# ── Proposal code block ───────────────────────────────────────────────
import pandas as pd
import numpy as np

# Load dataset 
# df = pd.read_csv("data/Churn_Modelling.csv")

# --- Shape & types
print("--- Dataset Shape ---")
# print(df.shape)

print("\n--- Column Types & Missing Values ---")
# print(df.info())

print("\n--- Summary Statistics ---")
# print(df.describe().T)

print("\n--- Churn Rate ---")
# print(df['Exited'].value_counts(normalize=True).rename({0: 'Stayed', 1: 'Churned'}))

## 4. Planned Methods

### 4a. Causal Inference

- [X] Causal graph / DAG (DoWhy)
- [X] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification

**Justification:**  
Financial metrics like account balances are heavily confounded by user demographic markers such as age (Age) and baseline earnings (EstimatedSalary). Wealthier or older clients naturally hold higher account balances and may display fundamentally different banking loyalty patterns. We will use the DoWhy package to map out an explicit Directed Acyclic Graph (DAG). By implementing a Backdoor Adjustment, we will formally condition on income and age to capture the true, unconfounded causal impact of financial variables (Balance, CreditScore) on the decision to exit the bank.

We'll construct a DAG using the DoWhy library with the following structure:
- EstimatedSalary → Balance, CreditScore, Exited
- Age → Balance, CreditScore, Exited
- Balance → Exited
- CreditScore → Exited

We'll then apply backdoor adjustment, conditioning on Age and EstimatedSalary, to estimate the average treatment effect (ATE) of Balance and CreditScore on churn. We'll validate this using DoWhy's refutation tests.

---

### 4b. Supervised Learning

- [ ] Linear / Ridge / Lasso regression
- [X] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [X] Decision Tree / Random Forest
- [ ] Neural network

**Justification:**  
Churn prediction is a binary classification problem, so regression isn't the right fit here. We'll start with logistic regression as a transparent baseline, it gives us interpretable log-odds coefficients for each feature, which connects nicely to our causal analysis. Then we'll train a Random Forest, which handles non-linear interactions naturally. We'll use stratified k-fold cross-validation throughout and  deal with the imbalance issue.

---

### 4c. Unsupervised Learning

- [X] K-Means clustering
- [ ] Hierarchical clustering
- [ ] Variational autoencoder
- [ ] GAN

**Justification:**  
Customers are not a homogeneous group. An inactive customer with a large balance is a very different churn risk than an active customer who holds multiple products. Before fitting a single global model, we want to understand whether the data naturally groups into distinct customer types.

We'll apply K-Means on a scaled feature set (Balance, CreditScore, NumOfProducts, IsActiveMember, Age) and choose k using the elbow method and silhouette score. The resulting cluster labels will then serve two purposes: first, as an additional feature in our supervised classifiers; second, as a stratification variable in our causal analysis. We want to know whether the causal effect of balance on churn is the same for all customer types, or whether it differs significantly by segment.

## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

## 5. Evaluation Strategy

**Predictive models:**  
Given the class imbalance, we'll report AUC-ROC as our primary metric, alongside F1-Score and Recall. Recall matters here because missing a true churner is more costly than a false alarm, a bank would rather send an unnecessary retention offer than lose a customer entirely.

**Causal estimates:**  
We'll run two DoWhy refutation tests:  
1. *Random common cause*  adds a random variable as a fake confounder; our estimated effect should not change significantly.  
2. *Placebo treatment*  replaces the treatment with a random variable; the estimated effect should collapse to near zero.  

If both tests pass, we have reasonable confidence the DAG is correctly specified.

**Clustering:**  
We'll select k using the elbow plot and silhouette coefficient. Cluster quality will also be evaluated descriptively, each cluster should be interpretable in terms of customer characteristics (e.g., "high-balance inactive customers").

**Cross-block connection:**  
The key evaluation question we want to answer at the end is: do the customer segments identified by K-Means moderate the causal effect of balance/credit score on churn? If so, that's a concrete, actionable finding for a bank.


## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | All Members | Data collection & cleaning |
| 2 | Leila | EDA |
| 3 | Leila | Causal inference block |
| 4 | Sara | Supervised learning block |
| 5 | Forough | Unsupervised / generative block |
| 6 | All Members| Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
